# Academic OSPO Readiness Survey — Analysis Notebook

This notebook produces the results in two parts.

**Part A — Completed responses.** Uses only fully completed responses, so its numbers match the spreadsheet (`analysis-helper.xlsx`). Use it to reconcile the two tools.

**Part B — Progressive view.** Recalculates the same insights over everyone who completed each part of the survey, whether or not they finished. A long survey loses people along the way, so reporting "84 people told us why open source matters" is a stronger, fairer signal than "12 finished everything." Each Part B figure is labelled with its own group size and includes people who stopped later, so these numbers are larger than Part A by design.

**How to run:** set `INPUT_PATH` below and run all cells. Export from LimeSurvey as **All responses** (Question code and question text, Strip HTML on, Full answers).

In [ ]:
import os, datetime
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = "your-survey-export.xlsx"   # <-- set this to your exported .xlsx file

plt.rcParams.update({"figure.dpi":110,"savefig.dpi":130,"axes.spines.top":False,"axes.spines.right":False,"font.size":10})
RUN = os.path.join("output", datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S"))
os.makedirs(RUN, exist_ok=True)
def save(fig, name): fig.savefig(os.path.join(RUN, name), bbox_inches="tight")
print("Outputs will be written to:", RUN)

## Mappings
Each item is tagged with its OSMM level and its primary Academic OSPO service, matching `../instructions/40-interpret.md`.

In [ ]:
mat=[("Q300a","L2",1),("Q300b","L2",4),("Q300c","L2",6),("Q300d","L2",13),("Q310a","L2",4),("Q310b","L2",4),("Q310c","L2",5),("Q310d","L4",7),("Q310e","L2",4),("Q310f","L2",13),("Q320a","L2",5),("Q320b","L2",5),("Q320c","L2",5),("Q330a","L2",6),("Q330b","L2",8),("Q330c","L2",6),("Q330d","L2",6),("Q330e","L2",5),("Q330f","L2",4),("Q340a","L3",8),("Q340b","L3",8),("Q340c","L3",8),("Q340d","L3",8),("Q340e","L3",3),("Q350a","L3",8),("Q350b","L3",10),("Q350c","L4",10),("Q350d","L3",12),("Q350e","L3",12),("Q360a","L4",7),("Q360b","L4",3),("Q360c","L4",10),("Q360d","L4",10),("Q360e","L4",2),("Q360f","L4",10),("Q360g","L4",11),("Q360h","L4",11),("Q370a","L3",8),("Q370b","L3",8),("Q370c","L5",1),("Q370d","L5",1),("Q370e","L5",9),("Q370f","L4",9),("Q370g","L4",9)]
sup=[("Q400a",4),("Q400b",5),("Q400c",5),("Q400d",5),("Q400e",9),("Q400f",13),("Q410a",6),("Q410b",6),("Q410c",6),("Q420a",3),("Q420b",7),("Q420c",3),("Q420d",10),("Q430a",2),("Q430b",2),("Q430c",2),("Q430d",2),("Q440a",9),("Q440b",9),("Q440c",9),("Q440d",9),("Q450a",11),("Q450b",11),("Q450c",11),("Q450d",11),("Q450e",11),("Q450f",11),("Q450g",11),("Q450h",11),("Q460a",8),("Q460b",12),("Q460c",12),("Q460d",12),("Q460e",10)]
def addcodes(items):
    out=[];cnt={}
    for it in items:
        b=it[0][:4];cnt[b]=cnt.get(b,0)+1;out.append((it,f"{b}[SQ{cnt[b]:03d}]"))
    return out
MAT=[(c,lv,sv,ec) for (c,lv,sv),ec in addcodes(mat)]
SUP=[(c,sv,ec) for (c,sv),ec in addcodes(sup)]
SVC={1:"OSPO Strategy & Vision",2:"Community of Practice",3:"Project Lifecycle Support & Maintenance",4:"License & IPR Compliance",5:"Supply-Chain Security & SBOM",6:"Choosing, Using & Procuring Open Source",7:"Open-Sourcing Guidance",8:"Internal Advocacy & Buy-In",9:"Sustainability & Funding Support",10:"Impact Measurement",11:"Developer Tooling & Infrastructure",12:"Outreach & Events",13:"Open Source AI Policy"}
AREA=[("Q300","Policy & OSPO"),("Q310","Compliance and risk"),("Q320","Security & supply chain"),("Q330","Selection, use & procurement"),("Q340","Open source contribution"),("Q350","Community and outreach"),("Q360","Hosting and maintenance"),("Q370","Leadership and strategy")]
WRITEINS=[("Q10[comment]","Unit / department"),("Q301.","Where support lives — Policy & OSPO"),("Q311.","Where support lives — Compliance and risk"),("Q321.","Where support lives — Security & supply chain"),("Q331.","Where support lives — Selection, use & procurement"),("Q341.","Where support lives — Contribution"),("Q351.","Where support lives — Community & outreach"),("Q361.","Where support lives — Hosting & maintenance"),("Q371.","Where support lives — Leadership & strategy"),("Q401.","Other support — Legal, compliance & risk"),("Q411.","Other support — Selection, use & procurement"),("Q421.","Other support — Project creation & maintenance"),("Q431.","Other support — Community & mentoring"),("Q441.","Other support — Funding & sustainability"),("Q451.","Other support — Tools & compute"),("Q461.","Other support — Advocacy & outreach"),("Q70.","Final feedback")]

## Load the export and define the groups

In [ ]:
df=pd.read_excel(INPUT_PATH); df.columns=[str(c) for c in df.columns]; cols=list(df.columns)
def find(prefix): return next((c for c in cols if c.startswith(prefix)), None)
N=len(df)
subm=find("submitdate")
def reached(prefix):
    col=find(prefix)
    return pd.Series([False]*N, index=df.index) if col is None else (df[col].notna() & ~df[col].astype(str).str.strip().isin(["","N/A"]))

# Measure how far each person got by the PAGE they reached (lastpage) and whether
# they submitted (submitdate) -- never by whether answer cells are filled. Every
# maturity/support question ends with a "None of the above" option; picking it makes
# LimeSurvey blank the other options to N/A, so content-based detection would wrongly
# count a valid response as "did not reach this section". Page map for this survey:
#   page 3 = confidence (end of Sections 1-2) | page 12 = last maturity question
#   (Section 3) | page 20 = last support question (Section 4) | page 25 = submitted.
lp=find("lastpage"); lpv=pd.to_numeric(df[lp],errors="coerce") if lp else pd.Series([np.nan]*N,index=df.index)
m_complete = (df[subm].notna() & (df[subm].astype(str).str.strip()!="") & (df[subm].astype(str)!="N/A")) if subm else pd.Series([True]*N, index=df.index)
m_p12 = lpv>=3      # completed Sections 1-2 (reached the confidence question)
m_p3  = lpv>=12     # completed Section 3 (reached the last maturity question)
m_p4  = lpv>=20     # completed Section 4 (reached the last support question)

dfc=df[m_complete]; d12=df[m_p12]; d3=df[m_p3]; d4=df[m_p4]
Nc,N12,N3,N4 = len(dfc),len(d12),len(d3),len(d4)
print(f"Loaded {N} responses. Completed parts 1-2: {N12} | completed part 3: {N3} | completed part 4: {N4} | finished survey: {Nc}")

def pct(code, frame):
    col=find(code); n=len(frame)
    return None if (col is None or n==0) else round((frame[col]=="Yes").sum()/n*100)
def maturity_df(frame):
    return pd.DataFrame([{"code":c,"level":lv,"svc":sv,"area":c[:4],"pct":pct(ec,frame)} for c,lv,sv,ec in MAT])
def by_level(mdf): return mdf.groupby("level")["pct"].mean().round(1)
def by_area(mdf): return pd.Series({name:round(mdf[mdf.area==code].pct.mean(),1) for code,name in AREA})
def demand_svc(frame):
    s=pd.DataFrame([{"svc":sv,"pct":pct(ec,frame)} for c,sv,ec in SUP])
    return pd.Series({k:(round(s[s.svc==k].pct.mean(),1) if (s.svc==k).any() else np.nan) for k in SVC})
def maturity_svc(mdf):
    return pd.Series({k:(round(mdf[mdf.svc==k].pct.mean(),1) if (mdf.svc==k).any() else np.nan) for k in SVC})
def dist(prefix,labels,frame):
    col=find(prefix); vc=frame[col].value_counts() if col else {}
    return pd.Series({l:int(vc.get(l,0)) for l in labels})
# Roles (Q10) are read straight from the data: institutions are told to customise Q10 in setup,
# so we count whatever answer labels actually appear instead of matching a fixed list.
def role_counts(prefix, frame):
    col=find(prefix)
    if not col: return pd.Series(dtype=int)
    s=frame[col].dropna().astype(str).str.strip()
    s=s[~s.isin(["","N/A"])]
    vc=s.value_counts(); vc.index.name=None
    return vc
IMP=["Not important","Somewhat important","Important","Critical"]
CONF=["Not confident","Somewhat confident","Confident","Very confident"]

# Part A — Completed responses (matches the spreadsheet)

Everything below uses only the **fully completed** responses, so the numbers reconcile with `analysis-helper.xlsx`.

In [ ]:
print(f"Part A is based on {Nc} fully completed responses (of {N} in the file).")
mA=maturity_df(dfc); blA=by_level(mA); baA=by_area(mA)
demA=demand_svc(dfc); matA=maturity_svc(mA)
svcA=pd.DataFrame({"maturity":matA,"demand":demA}); svcA.index=[SVC[i] for i in svcA.index]
svcA["gap"]=(svcA["demand"]-svcA["maturity"]).round(1)
print("\nMaturity by level:\n",blA.to_string())
print("\nServices by gap (demand - maturity):\n",svcA.sort_values("gap",ascending=False).to_string())
# data-quality flags (completed)
mat_cols=[find(ec) for _,_,_,ec in MAT if find(ec)]; sup_cols=[find(ec) for _,_,ec in SUP if find(ec)]
my=(dfc[mat_cols]=="Yes").sum(axis=1); sy=(dfc[sup_cols]=="Yes").sum(axis=1)
ticks_all=int(((my==len(mat_cols))|(sy==len(sup_cols))).sum()); ticks_none=int(((my==0)&(sy==0)).sum())
print(f"\nResponse patterns: ticked everything {ticks_all}, ticked nothing / 'None of the above' throughout {ticks_none}, of {Nc} completed")

In [ ]:
# Part A charts
fig,axes=plt.subplots(1,2,figsize=(11,3.2))
blA.plot.bar(ax=axes[0],color="#54A24B"); axes[0].set_title(f"Maturity by OSMM level (completed, n={Nc})"); axes[0].set_ylim(0,100); axes[0].set_ylabel("%")
baA.sort_values().plot.barh(ax=axes[1],color="#54A24B"); axes[1].set_title("Maturity by area"); axes[1].set_xlim(0,100)
plt.tight_layout(); save(fig,"A_maturity.png"); plt.show()
q=svcA.dropna(subset=["demand","maturity"])
fig,ax=plt.subplots(figsize=(10,7))
ax.scatter(q["maturity"],q["demand"],s=80,color="#4C78A8",zorder=3)
ax.axvline(q["maturity"].median(),color="#bbb",ls="--"); ax.axhline(q["demand"].median(),color="#bbb",ls="--")
ax.margins(0.18)   # auto-scale to the data with padding, so the dots spread out
ax.set_xlabel("Maturity %"); ax.set_ylabel("Demand %"); ax.set_title(f"Priority quadrant (completed, n={Nc}) — upper-left = act first")
labels=[ax.text(r["maturity"],r["demand"],nm,fontsize=8) for nm,r in q.iterrows()]
try:
    from adjustText import adjust_text
    adjust_text(labels,ax=ax,expand=(1.5,2.0),force_text=(0.6,1.0),arrowprops=dict(arrowstyle="-",color="#888",lw=0.6))
except Exception as e:
    print("adjustText not installed; labels may overlap. Add it with: pip install adjustText")
save(fig,"A_quadrant.png"); plt.show()
roleA=role_counts("Q10.",dfc); impA=dist("Q20.",IMP,dfc); confA=dist("Q22.",CONF,dfc)
fig,axes=plt.subplots(1,3,figsize=(13,3))
roleA[roleA>0].plot.barh(ax=axes[0],color="#4C78A8"); axes[0].set_title("Role")
impA.plot.bar(ax=axes[1],color="#F58518"); axes[1].set_title("Importance")
confA.plot.bar(ax=axes[2],color="#72B7B2"); axes[2].set_title("Confidence")
plt.suptitle(f"Respondents (completed, n={Nc})"); plt.tight_layout(); save(fig,"A_respondents.png"); plt.show()

In [ ]:
# Part A write-in comments (completed), verbatim, grouped by question
for key,topic in WRITEINS:
    col=find(key)
    if not col: continue
    vals=[str(v).strip() for v in dfc[col] if pd.notna(v) and str(v).strip() not in ("","N/A")]
    if vals:
        print(f"\n### {topic}  ({len(vals)})")
        for v in vals: print("  -", v)

# Part B — Progressive view (everyone who completed each part)

These are the same insights as Part A, recalculated over everyone who completed the relevant part, whether or not they finished the whole survey. The numbers are larger than Part A on purpose: someone who answered the first parts and stopped in part 3 still tells you who responded and why open source matters. Each figure below is labelled with the group it is based on.

In [ ]:
funnel=[("Completed parts 1-2",N12),("Completed part 3 (maturity)",N3),("Completed part 4 (support)",N4),("Finished the survey",Nc)]
for lab,v in funnel: print(f"  {lab:30s} {v}")
fig,ax=plt.subplots(figsize=(6.5,2.6)); ax.barh([f[0] for f in funnel][::-1],[f[1] for f in funnel][::-1],color="#4C78A8")
ax.set_xlabel("respondents"); ax.set_title("How far respondents got"); save(fig,"B_funnel.png"); plt.show()

In [ ]:
# Part B: who responded and why, over everyone who completed parts 1-2
roleB=role_counts("Q10.",d12); impB=dist("Q20.",IMP,d12); confB=dist("Q22.",CONF,d12)
reasons=[("Q21[SQ001]","Reduces vendor lock-in"),("Q21[SQ002]","Lowers costs"),("Q21[SQ003]","Enables collaboration"),("Q21[SQ004]","Transparency and trust"),("Q21[SQ005]","Accelerates innovation"),("Q21[SQ006]","Funder/regulatory requirements"),("Q21[SQ007]","Digital sovereignty / autonomy")]
reasonsB=pd.Series({lab:pct(code,d12) for code,lab in reasons})
print(f"Based on {N12} respondents who completed parts 1-2.\n")
print("Role:\n",roleB.to_string()); print("\nImportance:\n",impB.to_string()); print("\nWhy it matters (% of these respondents):\n",reasonsB.to_string())
fig,axes=plt.subplots(1,3,figsize=(13,3))
roleB[roleB>0].plot.barh(ax=axes[0],color="#4C78A8"); axes[0].set_title("Role")
impB.plot.bar(ax=axes[1],color="#F58518"); axes[1].set_title("Importance")
reasonsB.sort_values().plot.barh(ax=axes[2],color="#9D755D"); axes[2].set_title("Why (%)"); axes[2].set_xlim(0,100)
plt.suptitle(f"Who responded and why (completed parts 1-2, n={N12})"); plt.tight_layout(); save(fig,"B_who.png"); plt.show()

In [ ]:
# Part B: maturity, over everyone who completed part 3
mB=maturity_df(d3); blB=by_level(mB); baB=by_area(mB)
print(f"Based on {N3} respondents who completed part 3 (maturity).\n")
print("Maturity by level:\n",blB.to_string()); print("\nMaturity by area:\n",baB.to_string())
fig,axes=plt.subplots(1,2,figsize=(11,3.2))
blB.plot.bar(ax=axes[0],color="#54A24B"); axes[0].set_title(f"Maturity by level (part-3 group, n={N3})"); axes[0].set_ylim(0,100); axes[0].set_ylabel("%")
baB.sort_values().plot.barh(ax=axes[1],color="#54A24B"); axes[1].set_title("Maturity by area"); axes[1].set_xlim(0,100)
plt.tight_layout(); save(fig,"B_maturity.png"); plt.show()

In [ ]:
# Part B: demand over the part-4 group; gap = part-3 maturity minus part-4 demand
demB=demand_svc(d4); matB=maturity_svc(mB)
svcB=pd.DataFrame({"maturity (n=%d)"%N3:matB,"demand (n=%d)"%N4:demB}); svcB.index=[SVC[i] for i in svcB.index]
svcB["gap"]=(svcB.iloc[:,1]-svcB.iloc[:,0]).round(1)
print(f"Demand based on {N4} respondents who completed part 4; maturity from the {N3} who completed part 3.\n")
print(svcB.sort_values("gap",ascending=False).to_string())
fig,ax=plt.subplots(figsize=(7,4)); d=svcB.dropna(subset=[svcB.columns[1]]).sort_values(svcB.columns[1])
ax.barh(d.index,d.iloc[:,1],color="#E45756"); ax.set_title(f"Service demand (part-4 group, n={N4})"); ax.set_xlim(0,100); save(fig,"B_demand.png"); plt.show()

## Save tables and findings

In [ ]:
with pd.ExcelWriter(os.path.join(RUN,"summary_tables.xlsx")) as xw:
    blA.rename("maturity %").to_frame().to_excel(xw,sheet_name="A_by_level")
    baA.rename("maturity %").to_frame().to_excel(xw,sheet_name="A_by_area")
    svcA.to_excel(xw,sheet_name="A_services")
    blB.rename("maturity %").to_frame().to_excel(xw,sheet_name="B_by_level")
    svcB.to_excel(xw,sheet_name="B_services")
topA=svcA.dropna(subset=["gap"]).sort_values("gap",ascending=False).head(3)
lines=[f"# Findings — {os.path.basename(INPUT_PATH)}","",
 "## Part A — completed responses only (matches the spreadsheet)",
 f"- Respondents: {Nc} completed (of {N} in the file)",
 f"- Data-quality flags: ticks-everything {ticks_all}, ticks-nothing {ticks_none}",
 f"- Maturity by level: "+", ".join(f"{k} {v:.0f}%" for k,v in blA.items()),
 f"- Most mature area: {baA.idxmax()} ({baA.max():.0f}%); least: {baA.idxmin()} ({baA.min():.0f}%)",
 "- Priority services (largest gap): "+", ".join(f"{i} (gap {r.gap:.0f})" for i,r in topA.iterrows()),
 "",
 "## Part B — progressive view (more of your data)",
 f"- Completed parts 1-2: {N12} | completed part 3: {N3} | completed part 4: {N4} | finished: {Nc}",
 f"- Importance among the {N12} who did parts 1-2: "+", ".join(f"{k} {v}" for k,v in impB.items()),
 f"- Maturity by level among the {N3} who did part 3: "+", ".join(f"{k} {v:.0f}%" for k,v in blB.items()),
 f"- Top demand among the {N4} who did part 4: "+", ".join(f"{i} {r.iloc[1]:.0f}%" for i,r in svcB.dropna(subset=[svcB.columns[1]]).sort_values(svcB.columns[1],ascending=False).head(3).iterrows())]
open(os.path.join(RUN,"findings.md"),"w").write("\n".join(lines))
print("\n".join(lines)); print("\nSaved to",RUN,"->",sorted(os.listdir(RUN)))